# Multilingual Health Question Answering in Low-Resource African Languages

**Machine Learning Techniques I — Final Project**  
**Paulette Dushime | African Leadership University**

---

This notebook documents all 10 experiments conducted for the Zindi 
[Multilingual Health QA Challenge](https://zindi.africa/competitions/multilingual-health-question-answering-in-low-resource-african-languages-challenge).

| Section | Experiments | Approach |
|---|---|---|
| Part A | Exp 1 | TF-IDF retrieval baseline |
| Part B | Exp 2–3 | Generative fine-tuning (mT5-small) |
| Part C | Exp 4–5 | Semantic retrieval (MiniLM → LaBSE) |
| Part D | Exp 6–7 | Hybrid LaBSE + TF-IDF fallback |
| Part E | Exp 8 | Expanded retrieval pool (train+val) |
| Part F | Exp 9–10 | Selection-strategy ablations |

**Best leaderboard score:** 0.5483 (Experiment 8)  
See `docs/EXPERIMENT_LOG.md` for full results table.

---

## How to Run

**On Google Colab:**
1. Runtime → Change runtime type → T4 GPU
2. Mount Google Drive and upload the 4 dataset CSVs to `My Drive/multilingual_health_qa/`
3. Run all cells in order (Runtime → Run All)

**On Kaggle:**
1. Attach the dataset and update `DATA_DIR` in the Paths cell (Section 0)
2. Enable GPU accelerator in the session panel
3. Run all cells in order


## Section 0 — Environment Setup & Paths

In [ ]:
# GPU pinning — must run before any torch/transformers import
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'


In [ ]:
# Install packages
!pip install -q transformers sentencepiece accelerate datasets rouge-score sentence-transformers
print('Packages installed')


In [ ]:
# Verify CUDA — do not proceed past this cell unless it prints True
import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA version    : {torch.version.cuda}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')


In [ ]:
import re
import gc
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option('display.max_colwidth', 120)
print('Imports complete')


### Paths

Update `DATA_DIR` to match your environment:
- **Colab:** `Path('/content/drive/MyDrive/multilingual_health_qa')`
- **Kaggle:** `Path('/kaggle/input/datasets/paulettedushime/multilingual')` (or run the path-finder cell below)


In [ ]:
# Path finder — uncomment if unsure of the exact dataset path on Kaggle
# import os
# for root, dirs, files in os.walk('/kaggle/input'):
#     for f in files: print(os.path.join(root, f))

# Set your DATA_DIR here
# Colab:
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = Path('/content/drive/MyDrive/multilingual_health_qa')

# Kaggle (comment out Colab block above and uncomment below):
# DATA_DIR = Path('/kaggle/input/datasets/paulettedushime/multilingual')

TRAIN_PATH      = DATA_DIR / 'Train.csv'
TEST_PATH       = DATA_DIR / 'Test.csv'
VAL_PATH        = DATA_DIR / 'Val.csv'
SAMPLE_SUB_PATH = DATA_DIR / 'SampleSubmission.csv'

# Outputs always go to working directory
OUTPUT_DIR = Path('/content') if Path('/content').exists() else Path('/kaggle/working')

for path in [TRAIN_PATH, TEST_PATH, VAL_PATH]:
    print('OK' if path.exists() else 'MISSING', '-', path)


In [ ]:
# Load and clean data
train             = pd.read_csv(TRAIN_PATH)
test              = pd.read_csv(TEST_PATH)
val               = pd.read_csv(VAL_PATH)
sample_submission = pd.read_csv(SAMPLE_SUB_PATH)

ID_COL            = 'ID'
QUESTION_COL      = 'input'
ANSWER_COL        = 'output'
LANG_COL          = 'subset'

def clean_text(x):
    return '' if pd.isna(x) else str(x).strip()

train[QUESTION_COL] = train[QUESTION_COL].map(clean_text)
train[ANSWER_COL]   = train[ANSWER_COL].map(clean_text)
test[QUESTION_COL]  = test[QUESTION_COL].map(clean_text)
val[QUESTION_COL]   = val[QUESTION_COL].map(clean_text)
val[ANSWER_COL]     = val[ANSWER_COL].map(clean_text)

train = train[(train[QUESTION_COL] != '') & (train[ANSWER_COL] != '')].reset_index(drop=True)
val   = val[(val[QUESTION_COL] != '') & (val[ANSWER_COL] != '')].reset_index(drop=True)
test  = test[test[QUESTION_COL] != ''].reset_index(drop=True)

print(f'Train: {train.shape}  Val: {val.shape}  Test: {test.shape}')
print('\nLanguage distribution (train):')
print(train[LANG_COL].value_counts().to_string())


In [ ]:
# ROUGE evaluation utility — whitespace tokenizer for script-agnostic evaluation
from rouge_score import rouge_scorer

class WhitespaceTokenizer:
    def tokenize(self, text):
        return [] if text is None else str(text).strip().split()

def compute_rouge(predictions, references):
    scorer = rouge_scorer.RougeScorer(
        ['rouge1', 'rougeL'], tokenizer=WhitespaceTokenizer(), use_stemmer=False
    )
    r1, rl = [], []
    for pred, ref in zip(predictions, references):
        s = scorer.score(str(ref), str(pred))
        r1.append(s['rouge1'].fmeasure)
        rl.append(s['rougeL'].fmeasure)
    return {'rouge1_f1': float(np.mean(r1)), 'rougeL_f1': float(np.mean(rl))}

def make_submission(ids, predictions, output_path):
    clean_preds = [re.sub(r'<extra_id_\d+>', '', str(p)).strip() for p in predictions]
    sub = pd.DataFrame({
        'ID': ids, 'TargetRLF1': clean_preds,
        'TargetR1F1': clean_preds, 'TargetLLM': clean_preds
    })
    assert len(sub) == len(test)
    sub.to_csv(output_path, index=False, encoding='utf-8')
    print(f'Saved to {output_path} — shape {sub.shape}')
    return sub

print('ROUGE utility and submission helper ready')


---
## Part A — Experiment 1: TF-IDF Retrieval Baseline

**Change:** Character n-gram (3–5) TF-IDF retrieval, grouped per language subset.  
**Rationale:** Non-neural baseline using character-level features that work across
all scripts (Latin, Ge'ez) without language-specific tokenization.  
**Result:** Establishes the anchor score before any model training.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

class TfidfRetrievalAnswerer:
    """Character n-gram TF-IDF nearest-neighbour retrieval, per language subset."""

    def __init__(self, question_col, answer_col, group_col=None,
                 ngram_range=(3, 5), max_features=200_000):
        self.question_col = question_col
        self.answer_col   = answer_col
        self.group_col    = group_col
        self.ngram_range  = ngram_range
        self.max_features = max_features
        self.models       = {}
        self.global_model = None

    def _fit_single(self, df):
        vectorizer = TfidfVectorizer(
            analyzer='char_wb', ngram_range=self.ngram_range,
            min_df=1, max_features=self.max_features, lowercase=False
        )
        questions = df[self.question_col].fillna('').astype(str).tolist()
        answers   = df[self.answer_col].fillna('').astype(str).tolist()
        X  = vectorizer.fit_transform(questions)
        nn = NearestNeighbors(n_neighbors=1, metric='cosine')
        nn.fit(X)
        return {'vectorizer': vectorizer, 'nn': nn,
                'answers': np.array(answers, dtype=object)}

    def fit(self, df):
        self.global_model = self._fit_single(df)
        if self.group_col and self.group_col in df.columns:
            for group, sub in df.groupby(self.group_col):
                if len(sub) >= 2:
                    self.models[group] = self._fit_single(sub)
        print(f'Fitted global model + {len(self.models)} group models')
        return self

    def predict(self, df, question_col, group_col=None):
        outputs = []
        for _, row in df.iterrows():
            q     = clean_text(row[question_col])
            group = row[group_col] if group_col and group_col in df.columns else None
            model = self.models.get(group, self.global_model) if group else self.global_model
            Xq    = model['vectorizer'].transform([q])
            _, idx = model['nn'].kneighbors(Xq)
            outputs.append(model['answers'][idx[0][0]])
        return outputs

print('TfidfRetrievalAnswerer defined')


In [ ]:
# Experiment 1: TF-IDF retrieval — validation
tfidf_answerer = TfidfRetrievalAnswerer(
    question_col=QUESTION_COL, answer_col=ANSWER_COL, group_col=LANG_COL
).fit(train)

val_pred_tfidf = tfidf_answerer.predict(val, QUESTION_COL, LANG_COL)
metrics_tfidf  = compute_rouge(val_pred_tfidf, val[ANSWER_COL].tolist())

print('Experiment 1 — TF-IDF Retrieval Baseline')
print(f'  ROUGE-1 F1 : {metrics_tfidf["rouge1_f1"]:.4f}')
print(f'  ROUGE-L F1 : {metrics_tfidf["rougeL_f1"]:.4f}')

# Generate test predictions
test_pred_tfidf = tfidf_answerer.predict(test, QUESTION_COL, LANG_COL)
sub_tfidf = make_submission(test[ID_COL].values, test_pred_tfidf,
                             OUTPUT_DIR / 'submission_tfidf_baseline.csv')


---
## Part B — Experiments 2–3: Generative Fine-Tuning (mT5-small)

**Experiment 2:** mT5-small, 3 epochs  
**Experiment 3:** mT5-small, 6 epochs (epochs increased because val loss was still dropping)

**Key memory fixes applied (see report Section 5):**
- `optim='adafactor'` — avoids Adam's extra optimizer-state memory overhead
- `CUDA_VISIBLE_DEVICES='0'` — prevents DataParallel gather OOM on dual-GPU Kaggle sessions
- `generation_num_beams=1` — greedy decoding, far cheaper than beam search during eval
- `eval_accumulation_steps=1` — moves eval predictions off GPU incrementally

**Configuration switch:** change `FINETUNE_EPOCHS` below to switch between Exp 2 (3) and Exp 3 (6).


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
from datasets import Dataset

# Fine-tuning configuration
MODEL_NAME              = 'google/mt5-small'
FINETUNE_EPOCHS         = 3      # Change to 6 for Experiment 3
FINETUNE_BATCH_SIZE     = 8
FINETUNE_LEARNING_RATE  = 5e-4
FINETUNE_MAX_INPUT_LEN  = 128
FINETUNE_MAX_TARGET_LEN = 256
FINETUNE_VAL_SIZE       = 0.05
FINETUNE_OUTPUT_DIR     = str(OUTPUT_DIR / f'mt5-finetuned-{FINETUNE_EPOCHS}epochs')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
print(f'Model  : {MODEL_NAME}')
print(f'Epochs : {FINETUNE_EPOCHS}')


In [ ]:
# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model_llm = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)
model_llm = model_llm.to(DEVICE)
print(f'{MODEL_NAME} loaded — {sum(p.numel() for p in model_llm.parameters())/1e6:.0f}M params')


In [ ]:
def build_prompt(question, language=None):
    return str(question).strip()

def make_hf_dataset(df, question_col, answer_col):
    records = [{'prompt': build_prompt(str(r[question_col])), 'answer': str(r[answer_col])}
               for _, r in df.iterrows()]
    raw_ds = Dataset.from_list(records)
    def preprocess(examples):
        inputs = tokenizer(examples['prompt'], max_length=FINETUNE_MAX_INPUT_LEN,
                           truncation=True, padding=False)
        labels = tokenizer(text_target=examples['answer'],
                           max_length=FINETUNE_MAX_TARGET_LEN, truncation=True, padding=False)
        inputs['labels'] = [
            [(t if t != tokenizer.pad_token_id else -100) for t in seq]
            for seq in labels['input_ids']
        ]
        return inputs
    return raw_ds.map(preprocess, batched=True, remove_columns=['prompt', 'answer'])

train_ft, val_ft = train_test_split(train, test_size=FINETUNE_VAL_SIZE,
                                     random_state=SEED, stratify=train[LANG_COL])
hf_train = make_hf_dataset(train_ft, QUESTION_COL, ANSWER_COL)
hf_val   = make_hf_dataset(val_ft, QUESTION_COL, ANSWER_COL)
print(f'HF train: {len(hf_train)}  HF val: {len(hf_val)}')


In [ ]:
# Train
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model_llm,
                                        label_pad_token_id=-100, pad_to_multiple_of=8)
training_args = Seq2SeqTrainingArguments(
    output_dir=FINETUNE_OUTPUT_DIR,
    num_train_epochs=FINETUNE_EPOCHS,
    per_device_train_batch_size=FINETUNE_BATCH_SIZE,
    per_device_eval_batch_size=4,
    eval_accumulation_steps=1,
    generation_num_beams=1,
    learning_rate=FINETUNE_LEARNING_RATE,
    optim='adafactor',
    predict_with_generate=True,
    bf16=(DEVICE == 'cuda' and torch.cuda.is_bf16_supported()),
    fp16=(DEVICE == 'cuda' and not torch.cuda.is_bf16_supported()),
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    logging_steps=200,
    generation_max_length=FINETUNE_MAX_TARGET_LEN,
    report_to='none',
)
trainer = Seq2SeqTrainer(
    model=model_llm, args=training_args,
    train_dataset=hf_train, eval_dataset=hf_val,
    processing_class=tokenizer, data_collator=data_collator,
)
print(f'Starting fine-tuning — {FINETUNE_EPOCHS} epoch(s)...')
trainer.train()
print('Fine-tuning complete')


In [ ]:
# Generate and save submission for this fine-tuned model
def generate_answers_batch(questions, languages=None, batch_size=16):
    if languages is None:
        languages = [None] * len(questions)
    all_answers = []
    n_batches = (len(questions) + batch_size - 1) // batch_size
    for b in range(n_batches):
        start, end = b * batch_size, min((b + 1) * batch_size, len(questions))
        prompts = [build_prompt(q) for q in questions[start:end]]
        inputs  = tokenizer(prompts, return_tensors='pt', padding=True,
                             truncation=True, max_length=FINETUNE_MAX_INPUT_LEN).to(DEVICE)
        with torch.no_grad():
            outputs = model_llm.generate(**inputs, max_new_tokens=FINETUNE_MAX_TARGET_LEN,
                                          num_beams=1, no_repeat_ngram_size=3)
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        all_answers.extend([re.sub(r'<extra_id_\d+>', '', a).strip() for a in decoded])
        if (b + 1) % 20 == 0 or (b + 1) == n_batches:
            print(f'  batch {b+1}/{n_batches}')
    return all_answers

model_llm.eval()
test_pred_ft = generate_answers_batch(test[QUESTION_COL].tolist(), test[LANG_COL].tolist())

# Validation ROUGE
val_sample = val.sample(n=300, random_state=SEED)
val_pred_ft = generate_answers_batch(val_sample[QUESTION_COL].tolist())
metrics_ft  = compute_rouge(val_pred_ft, val_sample[ANSWER_COL].tolist())
print(f'Experiment {2 if FINETUNE_EPOCHS == 3 else 3} — mT5-small {FINETUNE_EPOCHS} epochs')
print(f'  ROUGE-1 F1 : {metrics_ft["rouge1_f1"]:.4f}')
print(f'  ROUGE-L F1 : {metrics_ft["rougeL_f1"]:.4f}')

out_name = f'submission_mt5small_{FINETUNE_EPOCHS}epochs.csv'
make_submission(test[ID_COL].values, test_pred_ft, OUTPUT_DIR / out_name)


---
## Part C — Experiments 4–5: Semantic Retrieval

**Experiment 4:** MiniLM (`paraphrase-multilingual-MiniLM-L12-v2`, 384-dim)  
**Experiment 5:** LaBSE (`sentence-transformers/LaBSE`, 768-dim) — best validated result

**Rationale:** TF-IDF only matches exact lexical overlap. Sentence embeddings
can match semantically similar questions even when phrased differently.  
LaBSE is purpose-built for cross-lingual alignment across 109 languages,
making it better suited than MiniLM for Akan, Luganda, and Amharic.

**Configuration switch:** change `EMBED_MODEL_NAME` below to switch between Exp 4 and 5.


In [ ]:
# Free fine-tuning model memory before loading embedding model
if 'model_llm' in dir():
    del model_llm
gc.collect()
torch.cuda.empty_cache()

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Experiment 4: EMBED_MODEL_NAME = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
# Experiment 5: EMBED_MODEL_NAME = 'sentence-transformers/LaBSE'
EMBED_MODEL_NAME = 'sentence-transformers/LaBSE'  # change to MiniLM for Exp 4

encoder = SentenceTransformer(EMBED_MODEL_NAME, device=DEVICE)
print(f'Loaded {EMBED_MODEL_NAME} on {DEVICE}')

def embed_texts(texts, batch_size=64):
    return encoder.encode(texts, batch_size=batch_size,
                           show_progress_bar=True, normalize_embeddings=True)

# Build train-only retrieval index
train_questions  = train[QUESTION_COL].tolist()
train_embeddings = embed_texts(train_questions)
print(f'Train embeddings: {train_embeddings.shape}')

subset_indices = {}
for subset_code in train[LANG_COL].unique():
    idx = np.where(train[LANG_COL].values == subset_code)[0]
    subset_indices[subset_code] = idx
print('Subsets:', {k: len(v) for k, v in subset_indices.items()})

def retrieve_answer(question_embedding, subset):
    """Top-1 cosine similarity retrieval within the same language subset."""
    idx = subset_indices.get(subset)
    if idx is None or len(idx) == 0:
        idx = np.arange(len(train_questions))
    sims = cosine_similarity(question_embedding.reshape(1, -1), train_embeddings[idx]).flatten()
    return train[ANSWER_COL].iloc[int(idx[int(np.argmax(sims))])]

print('Retrieval function ready')


In [ ]:
# Validate on 500-sample subset
val_sample = val.sample(n=min(500, len(val)), random_state=SEED).reset_index(drop=True)
val_q_emb  = embed_texts(val_sample[QUESTION_COL].tolist())

val_pred_sem = [
    retrieve_answer(val_q_emb[i], val_sample[LANG_COL].iloc[i])
    for i in range(len(val_sample))
]
metrics_sem = compute_rouge(val_pred_sem, val_sample[ANSWER_COL].tolist())
exp_num = 5 if 'LaBSE' in EMBED_MODEL_NAME else 4
print(f'Experiment {exp_num} — Semantic Retrieval ({EMBED_MODEL_NAME.split("/")[-1]})')
print(f'  ROUGE-1 F1 : {metrics_sem["rouge1_f1"]:.4f}')
print(f'  ROUGE-L F1 : {metrics_sem["rougeL_f1"]:.4f}')


In [ ]:
# Generate test predictions and save submission
test_q_emb = embed_texts(test[QUESTION_COL].tolist())
test_pred_sem = [
    retrieve_answer(test_q_emb[i], test[LANG_COL].iloc[i] if LANG_COL in test.columns else None)
    for i in range(len(test))
]
model_tag = 'labse' if 'LaBSE' in EMBED_MODEL_NAME else 'minilm'
make_submission(test[ID_COL].values, test_pred_sem,
                OUTPUT_DIR / f'submission_semantic_{model_tag}.csv')


---
## Part D — Experiments 6–7: Hybrid LaBSE + TF-IDF Fallback

**Experiment 6:** threshold = 0.5 — fallback to TF-IDF when LaBSE confidence is low  
**Experiment 7:** threshold = 0.3 — diagnostic test to confirm why Exp 6 showed no change

**Finding:** Fallback triggered 0/500 times at *both* thresholds, confirming LaBSE
is consistently confident on this dataset. Hybrid adds no value here.


In [ ]:
# Build TF-IDF fallback index (per language subset)
tfidf_models = {}
for subset_code in train[LANG_COL].unique():
    idx = subset_indices[subset_code]
    questions = [train_questions[i] for i in idx]
    vec    = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5),
                              max_features=50_000, lowercase=False)
    matrix = vec.fit_transform(questions)
    tfidf_models[subset_code] = {'vectorizer': vec, 'matrix': matrix, 'indices': idx}
print('TF-IDF fallback index ready')


In [ ]:
# Experiment 6 (threshold=0.5) and Experiment 7 (threshold=0.3)
# Change SIMILARITY_THRESHOLD below to switch between them

SIMILARITY_THRESHOLD = 0.5   # Experiment 6; change to 0.3 for Experiment 7
fallback_triggered   = 0

def retrieve_answer_hybrid(question_text, question_embedding, subset):
    global fallback_triggered
    idx  = subset_indices.get(subset, np.arange(len(train_questions)))
    sims = cosine_similarity(question_embedding.reshape(1, -1), train_embeddings[idx]).flatten()
    best_local = int(np.argmax(sims))
    best_sim   = float(sims[best_local])
    best_global = int(idx[best_local])
    if best_sim >= SIMILARITY_THRESHOLD:
        return train[ANSWER_COL].iloc[best_global]
    fallback_triggered += 1
    info = tfidf_models.get(subset)
    if info is None:
        return train[ANSWER_COL].iloc[best_global]
    tfidf_sims = cosine_similarity(info['vectorizer'].transform([question_text]),
                                    info['matrix']).flatten()
    return train[ANSWER_COL].iloc[int(info['indices'][int(np.argmax(tfidf_sims))])]

# Run on validation sample (reuses val_sample and val_q_emb from Part C)
val_pred_hybrid = [
    retrieve_answer_hybrid(val_sample[QUESTION_COL].iloc[i], val_q_emb[i],
                           val_sample[LANG_COL].iloc[i])
    for i in range(len(val_sample))
]
exp_num = 6 if SIMILARITY_THRESHOLD == 0.5 else 7
metrics_hybrid = compute_rouge(val_pred_hybrid, val_sample[ANSWER_COL].tolist())
print(f'Experiment {exp_num} — Hybrid (threshold={SIMILARITY_THRESHOLD})')
print(f'  Fallback triggered : {fallback_triggered}/{len(val_sample)}')
print(f'  ROUGE-1 F1         : {metrics_hybrid["rouge1_f1"]:.4f}')
print(f'  ROUGE-L F1         : {metrics_hybrid["rougeL_f1"]:.4f}')


---
## Part E — Experiment 8: Expanded Retrieval Pool (Train + Val Combined)

**Change:** Expand retrieval candidates from ~29.8K (train only) to ~36.5K (train + val).  
**Rationale:** More candidates increases the chance of a closer semantic match for test questions.  
**Note:** Val answers are safe to use here — we are generating *test* predictions,
not re-evaluating on val, so no leakage.  
**Result:** Public LB 0.5483 — best score across all experiments (+13% vs Exp 5).


In [ ]:
# Build combined train+val retrieval pool
combined_questions = train[QUESTION_COL].tolist() + val[QUESTION_COL].tolist()
combined_answers   = pd.concat([train[ANSWER_COL], val[ANSWER_COL]], ignore_index=True)
combined_subsets   = pd.concat([train[LANG_COL],   val[LANG_COL]],   ignore_index=True)

print(f'Combined pool: {len(combined_questions)} questions (train {len(train)} + val {len(val)})')

combined_embeddings = embed_texts(combined_questions)

combined_subset_indices = {}
for subset_code in combined_subsets.unique():
    idx = np.where(combined_subsets.values == subset_code)[0]
    combined_subset_indices[subset_code] = idx

def retrieve_answer_combined(question_embedding, subset):
    idx  = combined_subset_indices.get(subset, np.arange(len(combined_questions)))
    sims = cosine_similarity(question_embedding.reshape(1, -1), combined_embeddings[idx]).flatten()
    return combined_answers.iloc[int(idx[int(np.argmax(sims))])]

print('Combined retrieval function ready')


In [ ]:
# Generate test predictions with the combined pool (Experiment 8 — best submission)
test_q_emb_combined = embed_texts(test[QUESTION_COL].tolist())

test_pred_combined = [
    retrieve_answer_combined(test_q_emb_combined[i],
                              test[LANG_COL].iloc[i] if LANG_COL in test.columns else None)
    for i in range(len(test))
]
print(f'Generated {len(test_pred_combined)} predictions')
make_submission(test[ID_COL].values, test_pred_combined,
                OUTPUT_DIR / 'submission_labse_trainval_combined.csv')  # best submission


---
## Part F — Experiments 9–10: Selection-Strategy Ablations

**Experiment 9:** Top-3 candidates, select longest answer — *worse* (−13% ROUGE-1)  
**Experiment 10:** Top-3 candidates, select by highest similarity — *confirms Exp 5 is optimal*

**Conclusion:** Top-1-by-similarity is already the optimal selection strategy;
length-based selection discards the best match in favor of a less relevant longer answer.


In [ ]:
# Experiment 9 — top-3, select longest answer
def retrieve_top3_longest(question_embedding, subset):
    """Top-3 by similarity, select the longest answer. Train-only pool."""
    idx  = subset_indices.get(subset, np.arange(len(train_questions)))
    sims = cosine_similarity(question_embedding.reshape(1, -1), train_embeddings[idx]).flatten()
    top3_local  = np.argsort(sims)[-3:]
    top3_global = [int(idx[i]) for i in top3_local]
    best_global = max(top3_global, key=lambda g: len(str(train[ANSWER_COL].iloc[g])))
    return train[ANSWER_COL].iloc[best_global]

val_pred_top3 = [retrieve_top3_longest(val_q_emb[i], val_sample[LANG_COL].iloc[i])
                 for i in range(len(val_sample))]
metrics_top3  = compute_rouge(val_pred_top3, val_sample[ANSWER_COL].tolist())
print('Experiment 9 — Top-3, Select Longest')
print(f'  ROUGE-1 F1 : {metrics_top3["rouge1_f1"]:.4f}  (Exp 5 was 0.4407 — expect drop)')
print(f'  ROUGE-L F1 : {metrics_top3["rougeL_f1"]:.4f}')


In [ ]:
# Experiment 10 — top-3, select by similarity (confirmatory, should match Exp 5)
def retrieve_top3_by_sim(question_embedding, subset):
    """Top-3 by similarity, select highest-similarity one. Should equal top-1."""
    idx  = subset_indices.get(subset, np.arange(len(train_questions)))
    sims = cosine_similarity(question_embedding.reshape(1, -1), train_embeddings[idx]).flatten()
    top3_local  = np.argsort(sims)[-3:]
    best_local  = top3_local[-1]  # argsort ascending; last = highest
    return train[ANSWER_COL].iloc[int(idx[best_local])]

val_pred_top3_sim = [retrieve_top3_by_sim(val_q_emb[i], val_sample[LANG_COL].iloc[i])
                     for i in range(len(val_sample))]
metrics_top3_sim  = compute_rouge(val_pred_top3_sim, val_sample[ANSWER_COL].tolist())
print('Experiment 10 — Top-3, Select by Similarity (confirmatory)')
print(f'  ROUGE-1 F1 : {metrics_top3_sim["rouge1_f1"]:.4f}  (expect ~0.4407)')
print(f'  ROUGE-L F1 : {metrics_top3_sim["rougeL_f1"]:.4f}')
print('\nSmall discrepancy vs Exp 5 due to argsort/argmax tie-breaking on near-equal similarities.')


---
## Summary of All Experiments

| # | Approach | Local ROUGE-1 | Local ROUGE-L | Public LB |
|---|---|---|---|---|
| 1 | TF-IDF retrieval | — | — | — |
| 2 | mT5-small, 3 epochs | 0.2322 | 0.1767 | 0.2749 |
| 3 | mT5-small, 6 epochs | 0.2752 | 0.2087 | (not submitted) |
| 4 | Semantic retrieval, MiniLM | 0.3842 | 0.3335 | 0.3349 |
| 5 | Semantic retrieval, LaBSE | 0.4407 | 0.3953 | 0.4851 |
| 6 | Hybrid LaBSE+TF-IDF (0.5) | 0.4407 | 0.3953 | 0.4851 |
| 7 | Hybrid (0.3) | 0.4407 | 0.3953 | (not submitted) |
| 8 | LaBSE, train+val pool | — | — | **0.5483** |
| 9 | Top-3 longest | 0.3831 | 0.3288 | (not submitted) |
| 10 | Top-3 by similarity | 0.4398 | 0.3945 | (not submitted) |

**Best submission:** `submission_labse_trainval_combined.csv` (Experiment 8, LB: 0.5483)

See `docs/EXPERIMENT_LOG.md` for full reasoning, configuration details, and insights per experiment.
